# 📓 Notebook 3: Per-Label Threshold Tuning
**Nhóm Mù Công Nghệ** | Đề tài 22 | Tuần 4  
Mục tiêu: Tìm threshold tối ưu riêng cho mỗi nhãn, tối đa hóa F1-score  
Áp dụng cho: LR Baseline và DistilBERT

## 1. Import & cấu hình

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pickle, os, re, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    classification_report, hamming_loss, f1_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve, confusion_matrix,
    ConfusionMatrixDisplay
)

LABELS       = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("✅ Import xong")


## 2. Tải dữ liệu & chuẩn bị

In [ ]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z0-9\s!?.,\'\-]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df = pd.read_csv('train.csv')
df['clean_text'] = df['comment_text'].apply(preprocess)

X = df['clean_text'].values
y = df[LABELS].values

# Chia: 70% train | 10% val (dùng để tune threshold) | 20% test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.125, random_state=RANDOM_STATE)
# 0.125 × 0.8 = 0.1 tổng → train≈70%, val≈10%, test=20%

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")


## 3. TF-IDF & huấn luyện LR

In [ ]:
tfidf = TfidfVectorizer(max_features=50_000, ngram_range=(1,2),
                        sublinear_tf=True, min_df=3)
X_train_tf = tfidf.fit_transform(X_train)
X_val_tf   = tfidf.transform(X_val)
X_test_tf  = tfidf.transform(X_test)

lr = OneVsRestClassifier(
    LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=RANDOM_STATE),
    n_jobs=-1
)
lr.fit(X_train_tf, y_train)
print("✅ Huấn luyện xong LR")

# Lấy xác suất trên val và test
y_val_proba  = lr.predict_proba(X_val_tf)
y_test_proba = lr.predict_proba(X_test_tf)


## 4. Core: Per-Label Threshold Tuning

In [ ]:
def tune_threshold(y_true_col, y_proba_col, thresholds=None, metric='f1'):
    """
    Tìm threshold tối ưu cho 1 nhãn.
    metric: 'f1' (mặc định) hoặc 'recall' (ưu tiên Recall – nhãn quan trọng như threat)
    Trả về: best_threshold, best_score, scores_dict
    """
    if thresholds is None:
        thresholds = np.arange(0.10, 0.61, 0.01)
    best_t, best_score = 0.5, 0.0
    scores = []
    for t in thresholds:
        y_pred_t = (y_proba_col >= t).astype(int)
        if metric == 'f1':
            score = f1_score(y_true_col, y_pred_t, zero_division=0)
        else:
            from sklearn.metrics import recall_score
            score = recall_score(y_true_col, y_pred_t, zero_division=0)
        scores.append(score)
        if score > best_score:
            best_score = score
            best_t     = t
    return best_t, best_score, np.array(scores)

thresholds_grid = np.arange(0.10, 0.61, 0.01)
results = []

print(f"{'Nhãn':<16} {'Threshold*':<12} {'F1 (t*)':<10} {'F1 (t=0.5)':<12} {'Cải thiện'}")
print("-" * 60)

best_thresholds = {}
for i, lbl in enumerate(LABELS):
    best_t, best_f1, score_arr = tune_threshold(
        y_val[:, i], y_val_proba[:, i], thresholds_grid,
        metric='f1'
    )
    # F1 với threshold mặc định 0.5
    f1_default = f1_score(y_val[:, i], (y_val_proba[:, i] >= 0.5).astype(int), zero_division=0)
    improve    = best_f1 - f1_default
    best_thresholds[lbl] = round(float(best_t), 2)
    results.append({'label': lbl, 'threshold': round(float(best_t),2),
                    'f1_tuned': round(best_f1, 4), 'f1_default': round(f1_default, 4),
                    'improvement': round(improve, 4)})
    print(f"{lbl:<16} {best_t:<12.2f} {best_f1:<10.4f} {f1_default:<12.4f} +{improve:.4f}")

print("\nThreshold tối ưu:", best_thresholds)
df_results = pd.DataFrame(results)


## 5. Vẽ F1 theo threshold từng nhãn (F1-Threshold Curve)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
COLORS = ['steelblue','darkorange','green','red','purple','saddlebrown']

for i, (lbl, ax, c) in enumerate(zip(LABELS, axes.flatten(), COLORS)):
    _, _, scores = tune_threshold(y_val[:, i], y_val_proba[:, i], thresholds_grid)
    ax.plot(thresholds_grid, scores, color=c, lw=2)
    best_t = best_thresholds[lbl]
    best_s = scores[np.argmin(np.abs(thresholds_grid - best_t))]
    ax.axvline(best_t, color='crimson', linestyle='--', lw=1.5, label=f'Best t={best_t:.2f}')
    ax.axvline(0.5,    color='gray',   linestyle=':',  lw=1.2, label='Default t=0.5')
    ax.scatter([best_t], [best_s], color='crimson', s=80, zorder=5)
    ax.set_title(f'{lbl}  (best F1={best_s:.3f})', fontweight='bold')
    ax.set_xlabel('Threshold'); ax.set_ylabel('F1-score')
    ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_xlim(0.08, 0.62)

plt.suptitle('F1-score theo Threshold từng nhãn – LR Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('f1_threshold_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. PR Curve từng nhãn

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for i, (lbl, ax, c) in enumerate(zip(LABELS, axes.flatten(), COLORS)):
    pre, rec, thres = precision_recall_curve(y_val[:, i], y_val_proba[:, i])
    ap = average_precision_score(y_val[:, i], y_val_proba[:, i])
    ax.plot(rec, pre, color=c, lw=2)

    # Đánh dấu điểm threshold tối ưu
    best_t = best_thresholds[lbl]
    # Tìm vị trí gần nhất với best_t trong mảng thres (thres có N-1 phần tử)
    if len(thres) > 0:
        idx = np.argmin(np.abs(thres - best_t))
        ax.scatter([rec[idx]], [pre[idx]], color='crimson', s=100, zorder=5,
                   label=f't*={best_t:.2f}\n(P={pre[idx]:.2f}, R={rec[idx]:.2f})')
        ax.legend(fontsize=8)

    ax.set_title(f'{lbl}  (PR-AUC={ap:.3f})', fontweight='bold')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.grid(alpha=0.3); ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.05)

plt.suptitle('PR Curve từng nhãn – LR Model (điểm đỏ = threshold tối ưu)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('pr_curves_per_label.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Đánh giá LR + Threshold trên tập Test

In [ ]:
thresh_arr = np.array([best_thresholds[l] for l in LABELS])
y_pred_tuned = (y_test_proba >= thresh_arr).astype(int)
y_pred_base  = (y_test_proba >= 0.5).astype(int)

print("=== CLASSIFICATION REPORT (LR + Threshold Tuning) ===")
print(classification_report(y_test, y_pred_tuned, target_names=LABELS, digits=3))

def metrics_summary(y_true, y_pred, y_proba, name):
    return {
        'model'     : name,
        'macro_f1'  : round(f1_score(y_true, y_pred, average='macro',  zero_division=0), 4),
        'micro_f1'  : round(f1_score(y_true, y_pred, average='micro',  zero_division=0), 4),
        'hamming'   : round(hamming_loss(y_true, y_pred), 4),
        'roc_auc'   : round(roc_auc_score(y_true, y_proba, average='macro'), 4),
        'pr_auc'    : round(average_precision_score(y_true, y_proba, average='macro'), 4),
    }

res_base  = metrics_summary(y_test, y_pred_base,  y_test_proba, "LR Baseline (t=0.5)")
res_tuned = metrics_summary(y_test, y_pred_tuned, y_test_proba, "LR + Threshold Tuning")
df_compare = pd.DataFrame([res_base, res_tuned])
print("\n=== SO SÁNH BASELINE vs THRESHOLD TUNING ===")
print(df_compare.to_string(index=False))


## 8. Confusion Matrix sau Threshold Tuning

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, (lbl, ax) in enumerate(zip(LABELS, axes.flatten())):
    cm = confusion_matrix(y_test[:, i], y_pred_tuned[:, i])
    sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', ax=ax,
                xticklabels=['Not', lbl], yticklabels=['Not', lbl])
    ax.set_title(f'Confusion Matrix – {lbl}\n(threshold={best_thresholds[lbl]})', fontweight='bold')
    ax.set_xlabel('Predicted label'); ax.set_ylabel('True label')
plt.suptitle('LR + Threshold Tuning – Confusion Matrix 6 nhãn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('multilabel_confusion_matrices_threshold.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Biểu đồ so sánh F1 trước và sau tuning

In [ ]:
f1_before = [f1_score(y_test[:, i], y_pred_base[:, i],  zero_division=0) for i in range(6)]
f1_after  = [f1_score(y_test[:, i], y_pred_tuned[:, i], zero_division=0) for i in range(6)]

x = np.arange(len(LABELS)); w = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - w/2, f1_before, w, label='LR Baseline (t=0.5)', color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + w/2, f1_after,  w, label='LR + Threshold Tuning', color='darkorange', alpha=0.85)
for bar in bars1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.2f}', ha='center', fontsize=9)
for bar in bars2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.2f}', ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=20)
ax.set_title('F1-score trước và sau Per-Label Threshold Tuning', fontweight='bold')
ax.set_ylabel('F1-score'); ax.set_ylim(0, 1.0); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('f1_before_after_tuning.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Lưu kết quả

In [ ]:
os.makedirs('results', exist_ok=True)
df_results.to_csv('results/threshold_results.csv', index=False)
pd.DataFrame([res_tuned]).to_csv('results/metrics_lr_threshold.csv', index=False)
import json
with open('results/best_thresholds.json', 'w') as f:
    json.dump(best_thresholds, f, indent=2)
print("✅ Đã lưu:")
print("   results/threshold_results.csv")
print("   results/metrics_lr_threshold.csv")
print("   results/best_thresholds.json")
print("\nThreshold tối ưu:", best_thresholds)
